# VitalPredict — Clinical Dataset Preprocessing & Feature Engineering Pipeline 🔮

This notebook contains the comprehensive data preprocessing, imputation, and normalization pipeline for **VitalPredict** (an Explainable Personal Health Decision Support Platform). 

We preprocess three clinical datasets representing key health dimensions:
1. **Sleep Health and Lifestyle Dataset:** Evaluates sleep quality, stress levels, and daily habits.
2. **Heart Disease Dataset:** Tracks cardiovascular biomarkers and structural heart indicators.
3. **Pima Indians Diabetes Dataset:** Focuses on metabolic indicators and glycemic thresholds.

---

## 1. Setup & Environment Configuration

We load standard scientific computing libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) and preprocessing helpers from `scikit-learn`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Settings for visualization
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

--- 
## 2. Dataset 1: Sleep Health & Lifestyle Preprocessing

### Step 2.1: Load Raw Dataset & Inspect Structure

In [ ]:
sleep_df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
print(f"Sleep Dataset Shape: {sleep_df.shape}")
sleep_df.head()

### Step 2.2: Handle Missing Values & Null Values

Let's check for missing entries. In the `Sleep Disorder` column, missing values (`NaN`) actually represent the absence of a diagnosed sleep disorder. We will impute these with the string `'None'` to maintain a clean categorical model.

In [ ]:
print("Missing values before imputation:")
print(sleep_df.isnull().sum())

# Impute 'Sleep Disorder' nulls with 'None'
sleep_df['Sleep Disorder'] = sleep_df['Sleep Disorder'].fillna('None')

print("\nMissing values after imputation:")
print(sleep_df.isnull().sum())

### Step 2.3: Parse Complex String Features (Blood Pressure Split)

The `Blood Pressure` column is in `Systolic/Diastolic` format (e.g. `120/80`). To feed this to numerical models, we split this into separate `bp_systolic` and `bp_diastolic` numerical attributes.

In [ ]:
sleep_df[['bp_systolic', 'bp_diastolic']] = sleep_df['Blood Pressure'].str.split('/', expand=True).astype(int)
sleep_df.drop('Blood Pressure', axis=1, inplace=True)
sleep_df.head()

### Step 2.4: Handle Duplicates

We drop the `Person ID` column since it is a primary key that contains no features, check for row duplicates, and remove them.

In [ ]:
sleep_df.drop('Person ID', axis=1, inplace=True)

duplicates = sleep_df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    sleep_df.drop_duplicates(inplace=True)
    print(f"Duplicates removed. New shape: {sleep_df.shape}")

### Step 2.5: Descriptive Statistics & Data Range Analysis

In [ ]:
sleep_df.describe()

### Step 2.6: Categorical Variable Encoding

We perform one-hot encoding on `Gender` and `Occupation`, ordinal map `BMI Category`, and label encode `Sleep Disorder`.

In [ ]:
# Ordinal Map BMI Category
bmi_mapping = {"Normal": 0, "Normal Weight": 0, "Overweight": 1, "Obese": 2}
sleep_df['BMI Category'] = sleep_df['BMI Category'].map(bmi_mapping)

# One-hot encode Gender and Occupation
sleep_encoded_df = pd.get_dummies(sleep_df, columns=['Gender', 'Occupation'], drop_first=True)

# Label encode Sleep Disorder
disorder_mapping = {"None": 0, "Insomnia": 1, "Sleep Apnea": 2}
sleep_encoded_df['Sleep Disorder'] = sleep_encoded_df['Sleep Disorder'].map(disorder_mapping)

sleep_encoded_df.head()

### Step 2.7: Feature Normalization

We scale the numerical fields between `0` and `1` using `MinMaxScaler` to ensure models do not weight features with larger integer scales heavily.

In [ ]:
scaler = MinMaxScaler()
num_cols = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 
            'Stress Level', 'Heart Rate', 'Daily Steps', 'bp_systolic', 'bp_diastolic']

sleep_scaled_df = sleep_encoded_df.copy()
sleep_scaled_df[num_cols] = scaler.fit_transform(sleep_encoded_df[num_cols])
sleep_scaled_df.head()

### Step 2.8: Correlation Matrix Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
correlation_matrix = sleep_scaled_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix - Sleep Health & Lifestyle (Processed)')
plt.tight_layout()
plt.savefig('sleep_correlation_heatmap.png')
plt.show()

--- 
## 3. Dataset 2: Heart Disease Preprocessing

### Step 3.1: Load Raw Dataset & Inspect Structure

In [ ]:
heart_df = pd.read_csv('heart.csv')
print(f"Heart Dataset Shape: {heart_df.shape}")
heart_df.head()

### Step 3.2: Check for Missing Values & Duplicates

In [ ]:
print("Missing Values:")
print(heart_df.isnull().sum())

heart_duplicates = heart_df.duplicated().sum()
print(f"\nDuplicate rows: {heart_duplicates}")
if heart_duplicates > 0:
    heart_df.drop_duplicates(inplace=True)
    print(f"Dropped duplicates. New shape: {heart_df.shape}")

### Step 3.3: Descriptive Statistics

In [ ]:
heart_df.describe()

### Step 3.4: Standardization of Numerical Attributes

We use `StandardScaler` to zero-center and scale numerical variables representing arterial pressure (`trestbps`), cholesterol (`chol`), max heart rate (`thalach`), and ST depression (`oldpeak`).

In [ ]:
scaler_std = StandardScaler()
num_heart_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

heart_processed_df = heart_df.copy()
heart_processed_df[num_heart_cols] = scaler_std.fit_transform(heart_df[num_heart_cols])
heart_processed_df.head()

### Step 3.5: Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(heart_processed_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix - Heart Disease Dataset (Processed)')
plt.tight_layout()
plt.savefig('heart_correlation_heatmap.png')
plt.show()

--- 
## 4. Dataset 3: Pima Indians Diabetes Preprocessing

### Step 4.1: Load Raw Dataset & Inspect Structure

In [ ]:
diabetes_df = pd.read_csv('diabetes.csv')
print(f"Diabetes Dataset Shape: {diabetes_df.shape}")
diabetes_df.head()

### Step 4.2: Handle Biologically Impossible Zeros (Imputation)

In the Pima Indians dataset, values of `0` in columns like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` represent missing data since a living patient cannot survive with zero blood pressure, zero BMI, or zero glucose. We replace these zeros with `NaN` and impute them using their respective median values based on class target (`Outcome`) groupings.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("Zero value counts before imputation:")
for col in zero_cols:
    print(f"{col}: {(diabetes_df[col] == 0).sum()} rows")

# Replace zeros with NaN
for col in zero_cols:
    diabetes_df[col] = diabetes_df[col].replace(0, np.nan)

# Impute missing values using median grouped by target Outcome class
for col in zero_cols:
    diabetes_df[col] = diabetes_df[col].fillna(diabetes_df.groupby('Outcome')[col].transform('median'))

print("\nZero value counts after imputation:")
for col in zero_cols:
    print(f"{col}: {(diabetes_df[col] == 0).sum()} rows")

### Step 4.3: Check for Duplicates

In [ ]:
diabetes_duplicates = diabetes_df.duplicated().sum()
print(f"Duplicate rows: {diabetes_duplicates}")

### Step 4.4: Descriptive Statistics

In [ ]:
diabetes_df.describe()

### Step 4.5: Feature Normalization

We apply `MinMaxScaler` scaling to bounds of `0` and `1` on all numerical features.

In [ ]:
scaler_diab = MinMaxScaler()
diab_num_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

diabetes_processed_df = diabetes_df.copy()
diabetes_processed_df[diab_num_cols] = scaler_diab.fit_transform(diabetes_df[diab_num_cols])
diabetes_processed_df.head()

### Step 4.6: Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(diabetes_processed_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix - Pima Indians Diabetes (Processed)')
plt.tight_layout()
plt.savefig('diabetes_correlation_heatmap.png')
plt.show()

--- 
## 5. Cleaned Dataset Export

We export the processed, standardized, and clean dataset dataframes into CSV files which are ready to feed into prediction algorithms (XGBoost, SHAP Explainer cores) on the platform.

In [ ]:
sleep_scaled_df.to_csv('cleaned_sleep_health.csv', index=False)
heart_processed_df.to_csv('cleaned_heart_disease.csv', index=False)
diabetes_processed_df.to_csv('cleaned_diabetes.csv', index=False)

print("CSV Exports successfully generated:")
print("1. cleaned_sleep_health.csv")
print("2. cleaned_heart_disease.csv")
print("3. cleaned_diabetes.csv")